In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 3.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import cv2
import joblib
import matplotlib.pyplot as plt

from ultralytics import YOLO
from google.colab import files

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
yolo_model = YOLO("yolo26n.pt")

print("YOLO model loaded successfully!")

YOLO model loaded successfully!


In [ ]:
uploaded_ml = files.upload()

Saving queueless_rf_model.pkl to queueless_rf_model.pkl


In [ ]:
ml_model_path = list(uploaded_ml.keys())[0]

rf_model = joblib.load(ml_model_path)

print("ML model loaded successfully!")

In [ ]:
uploaded_video = files.upload()

In [ ]:
video_path = list(uploaded_video.keys())[0]

print("Video loaded:", video_path)

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video")
else:
    print("Video opened successfully!")

In [ ]:
ret, frame = cap.read()

if ret:
    print("Frame captured successfully!")
    print("Frame shape:", frame.shape)
else:
    print("Could not read frame")

In [ ]:
results = yolo_model(frame)

print("YOLO detection completed!")

In [ ]:
annotated_frame = results[0].plot()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
from datetime import datetime

now = datetime.now()

current_hour = now.hour
current_day = now.strftime("%A")
is_weekend = now.weekday() >= 5

print("Hour:", current_hour)
print("Day:", current_day)
print("Weekend:", is_weekend)

In [ ]:
# ==========================================
# WHICH COUNTERS DID THE ML MODEL LEARN?
# ==========================================
try:
    ohe = rf_model.named_steps["preprocessor"].named_transformers_["categorical"]
    cat_cols = rf_model.named_steps["preprocessor"].transformers_[0][2]
    known = dict(zip(cat_cols, ohe.categories_))
    print("Counters (WorkstationGroupID) known to the ML model:", list(known["WorkstationGroupID"]))
    print("Days known to the ML model:", list(known["day_of_week"]))
except Exception as e:
    print("Could not introspect pipeline (that's OK, continue):", e)

In [ ]:
# ==========================================
# VIDEO PROPERTIES
# ==========================================
cap = cv2.VideoCapture(video_path)
FPS    = cap.get(cv2.CAP_PROP_FPS) or 24.0
WIDTH  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"{WIDTH}x{HEIGHT} @ {FPS:.1f} fps")

# ==========================================
# QUEUE ROIs — as FRACTIONS of frame width/height (0 to 1)
# so they work at any resolution. Each is a trapezoid that
# widens toward the camera and reaches the bottom of the frame.
# ADJUST THESE after seeing the preview below.
# ==========================================
QUEUE_ROIS_NORM = {
    "Queue 1": [(0.02, 0.35), (0.30, 0.35), (0.33, 1.00), (0.00, 1.00)],
    "Queue 2": [(0.35, 0.35), (0.63, 0.35), (0.66, 1.00), (0.34, 1.00)],
    "Queue 3": [(0.68, 0.35), (0.98, 0.35), (1.00, 1.00), (0.67, 1.00)],
}
LANE_NAMES  = list(QUEUE_ROIS_NORM.keys())
LANE_COLORS = {"Queue 1": (255, 0, 0), "Queue 2": (0, 165, 255), "Queue 3": (0, 200, 0)}

def to_pixels(norm_pts, w, h):
    return (np.array(norm_pts, dtype=np.float32) * [w, h]).astype(np.int32)

QUEUE_ROIS = {name: to_pixels(pts, WIDTH, HEIGHT) for name, pts in QUEUE_ROIS_NORM.items()}

def assign_lane(foot_point):
    for name, poly in QUEUE_ROIS.items():
        if cv2.pointPolygonTest(poly, foot_point, False) >= 0:
            return name
    return None

In [ ]:
cap = cv2.VideoCapture(video_path)
ok, preview_frame = cap.read()
cap.release()

preview = preview_frame.copy()
for name, poly in QUEUE_ROIS.items():
    cv2.polylines(preview, [poly], True, LANE_COLORS[name], 3)
    lx, ly = int(poly[0][0]) + 5, int(poly[0][1]) - 10
    cv2.putText(preview, name, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 1.0, LANE_COLORS[name], 3)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Adjust QUEUE_ROIS_NORM above until each box hugs one lane only")
plt.show()

In [ ]:
# ==========================================
# AUTOMATIC YOLO QUEUE COUNTING (fixed)
# Uses foot point (not box center) and takes the MEDIAN count
# across the whole clip per lane, instead of just the last frame.
# ==========================================
from collections import deque

SMOOTH_SEC = 1.0
smooth_f = max(1, int(SMOOTH_SEC * FPS))
history = {n: deque(maxlen=smooth_f) for n in LANE_NAMES}
frame_history = {n: [] for n in LANE_NAMES}  # every smoothed value, for the final median

cap = cv2.VideoCapture(video_path)
idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo_model(frame, verbose=False)
    result = results[0]

    raw = {n: 0 for n in LANE_NAMES}

    if result.boxes is not None:
        for box in result.boxes:
            cls = int(box.cls[0])
            if cls != 0:   # class 0 = person
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

            # FOOT POINT: bottom-center of the box, where the person is
            # actually standing, not the box center (which floats into
            # the neighbouring lane on an angled camera).
            fx, fy = int((x1 + x2) / 2), int(y2)

            lane = assign_lane((float(fx), float(fy)))
            if lane is not None:
                raw[lane] += 1

    for n in LANE_NAMES:
        history[n].append(raw[n])
        smoothed = int(np.median(history[n]))
        frame_history[n].append(smoothed)

    idx += 1

cap.release()

# Final "current" count per lane = median smoothed count over the whole clip
queue_counts = {n: int(np.median(frame_history[n])) for n in LANE_NAMES}

print(f"Processed {idx} frames.")
print("========== CV QUEUE COUNTS ==========")
for q, c in queue_counts.items():
    print(f"{q}: {c} people")

In [ ]:
from datetime import datetime

# Map each detected lane to the counter ID the ML model was trained on.
# Edit this to match your real store layout / the IDs printed above.
queue_to_counter = {
    "Queue 1": 1,
    "Queue 2": 8,
    # "Queue 3": 8,   # example: if you only have 2 physical counters trained,
                       # point extra lanes at the closest matching counter
}

avg_art_num = 15.528563291255535   # replace with a live estimate if you have one

now = datetime.now()
current_hour = now.hour
current_day = now.strftime("%A")
is_weekend = now.weekday() >= 5

counter_predictions = {}
for counter_id in set(queue_to_counter.values()):
    input_data = pd.DataFrame({
        "ArtNum": [avg_art_num],
        "hour": [current_hour],
        "day_of_week": [current_day],
        "is_weekend": [is_weekend],
        "WorkstationGroupID": [counter_id]
    })
    predicted_time = rf_model.predict(input_data)[0]
    counter_predictions[counter_id] = predicted_time
    print(f"Counter {counter_id}: {predicted_time:.2f} sec service time "
          f"({predicted_time/60:.2f} min)")

In [ ]:
wait_times = {}

for queue, counter_id in queue_to_counter.items():
    people = queue_counts[queue]
    service_time = counter_predictions[counter_id]
    estimated_wait = people * service_time
    wait_times[queue] = estimated_wait

    print(f"{queue} (Counter {counter_id}): {people} people × "
          f"{service_time:.2f} sec = {estimated_wait:.2f} sec "
          f"({estimated_wait/60:.2f} min)")

recommended_queue = min(wait_times, key=wait_times.get)
recommended_wait = wait_times[recommended_queue]

print("\n========== QUEUELESS RECOMMENDATION ==========")
print(f"Recommended: {recommended_queue} (Counter {queue_to_counter[recommended_queue]})")
print(f"Estimated Wait: {recommended_wait/60:.2f} minutes")

In [ ]:
!pip install -q streamlit
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.5 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [ ]:
!pip install pyngrok -q


In [ ]:
%%writefile app.py
"""


QueueLess — Streamlit UI
=========================
One app, two views:
  - Customer View   : "Go to Counter X, ~Y min wait"
  - Manager Dashboard: live counts, service-time predictions, alerts

Run locally:
    pip install -r requirements.txt
    streamlit run app.py

Data sources (pick one in the sidebar):
  1. Demo / manual  — move sliders to simulate queue lengths (no model needed
     to try the UI, but predictions need your trained RF model).
  2. Upload RF model (.pkl/.joblib) — the same file your Colab notebook
     produces with joblib.dump(rf_model, "rf_model.pkl").
  3. (Advanced, optional) Upload a video + let YOLO count people live. This
     needs `ultralytics` + `opencv-python` installed and is slow on a CPU-only
     machine — it's meant for a local demo, not a lightweight cloud deploy.
"""

import streamlit as st
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

st.set_page_config(page_title="QueueLess", page_icon="🛒", layout="wide")

# ----------------------------------------------------------------------
# Session state defaults
# ----------------------------------------------------------------------
if "rf_model" not in st.session_state:
    st.session_state.rf_model = None
if "queue_names" not in st.session_state:
    st.session_state.queue_names = ["Queue 1", "Queue 2", "Queue 3"]
if "queue_to_counter" not in st.session_state:
    st.session_state.queue_to_counter = {"Queue 1": 1, "Queue 2": 8, "Queue 3": 8}
if "queue_counts" not in st.session_state:
    st.session_state.queue_counts = {"Queue 1": 3, "Queue 2": 5, "Queue 3": 2}

# ----------------------------------------------------------------------
# Core logic (mirrors the notebook: CV count x RF service time = wait)
# ----------------------------------------------------------------------
def predict_service_times(rf_model, counter_ids, art_num, hour, day, is_weekend):
    """Run the trained RandomForest pipeline for each distinct counter id."""
    preds = {}
    for c in counter_ids:
        row = pd.DataFrame({
            "ArtNum": [art_num],
            "hour": [hour],
            "day_of_week": [day],
            "is_weekend": [is_weekend],
            "WorkstationGroupID": [c],
        })
        preds[c] = float(rf_model.predict(row)[0])
    return preds


def compute_wait_times(queue_counts, queue_to_counter, service_times):
    waits = {}
    for queue, counter_id in queue_to_counter.items():
        people = queue_counts.get(queue, 0)
        service_time = service_times.get(counter_id)
        if service_time is None:
            continue
        waits[queue] = people * service_time
    return waits


def known_counters(rf_model):
    """Introspect the pipeline to see which WorkstationGroupIDs it was trained on."""
    try:
        ohe = rf_model.named_steps["preprocessor"].named_transformers_["categorical"]
        cat_cols = rf_model.named_steps["preprocessor"].transformers_[0][2]
        cats = dict(zip(cat_cols, ohe.categories_))
        return list(cats.get("WorkstationGroupID", []))
    except Exception:
        return None


# ----------------------------------------------------------------------
# Sidebar — configuration
# ----------------------------------------------------------------------
with st.sidebar:
    st.title("🛒 QueueLess")
    st.caption("Smart checkout counter recommendation")

    st.header("1. ML model")
    model_file = st.file_uploader("Upload trained RF model (.pkl / .joblib)", type=["pkl", "joblib"])
    if model_file is not None:
        try:
            st.session_state.rf_model = joblib.load(model_file)
            st.success("Model loaded.")
        except Exception as e:
            st.error(f"Could not load model: {e}")

    rf_model = st.session_state.rf_model
    if rf_model is not None:
        counters_seen = known_counters(rf_model)
        if counters_seen:
            st.caption(f"Counters this model was trained on: {counters_seen}")

    st.header("2. Queues & counters")
    n_queues = st.number_input("Number of queues", min_value=1, max_value=8, value=len(st.session_state.queue_names))
    queue_names = [f"Queue {i+1}" for i in range(n_queues)]
    st.session_state.queue_names = queue_names

    st.caption("Map each queue to the checkout counter ID your model was trained on.")
    queue_to_counter = {}
    for q in queue_names:
        default = st.session_state.queue_to_counter.get(q, 1)
        queue_to_counter[q] = st.number_input(f"{q} → Counter ID", value=int(default), step=1, key=f"map_{q}")
    st.session_state.queue_to_counter = queue_to_counter

    st.header("3. Live queue counts")
    st.caption("From your CV/YOLO pipeline. Enter manually here for a demo, or wire in automatically (see bottom of page).")
    queue_counts = {}
    for q in queue_names:
        default = st.session_state.queue_counts.get(q, 3)
        queue_counts[q] = st.slider(f"{q}: people waiting", 0, 20, int(default), key=f"count_{q}")
    st.session_state.queue_counts = queue_counts

    st.header("4. Context")
    avg_art_num = st.number_input("Avg. items per basket", value=15.5, step=0.5)
    use_now = st.checkbox("Use current date/time", value=True)
    if use_now:
        now = datetime.now()
        hour, day, is_weekend = now.hour, now.strftime("%A"), now.weekday() >= 5
    else:
        hour = st.slider("Hour of day", 0, 23, 18)
        day = st.selectbox("Day", ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
        is_weekend = day in ("Saturday", "Sunday")

    alert_threshold = st.slider("Manager alert threshold (people)", 1, 20, 6)

# ----------------------------------------------------------------------
# Run predictions
# ----------------------------------------------------------------------
wait_times, service_times, error = {}, {}, None

if rf_model is None:
    error = "Upload your trained RF model in the sidebar to get real predictions. Showing layout with placeholder numbers."
    # Placeholder so the UI still renders meaningfully before a model is loaded
    service_times = {c: 40.0 for c in set(queue_to_counter.values())}
else:
    try:
        service_times = predict_service_times(
            rf_model, set(queue_to_counter.values()), avg_art_num, hour, day, is_weekend
        )
    except Exception as e:
        error = f"Prediction failed: {e}"

wait_times = compute_wait_times(queue_counts, queue_to_counter, service_times)

recommended_queue = min(wait_times, key=wait_times.get) if wait_times else None

# ----------------------------------------------------------------------
# Main area — two views
# ----------------------------------------------------------------------
st.title("QueueLess")
if error:
    st.warning(error)

tab_customer, tab_manager = st.tabs(["🛍️ Customer View", "📊 Manager Dashboard"])

# ---------------- Customer View ----------------
with tab_customer:
    st.subheader("Where should I check out?")

    if recommended_queue:
        rec_wait_min = wait_times[recommended_queue] / 60
        st.markdown(
            f"""
            <div style="padding:2rem;border-radius:1rem;background:#16a34a;color:white;text-align:center;">
                <div style="font-size:1.1rem;opacity:0.9;">Recommended</div>
                <div style="font-size:3rem;font-weight:800;">{recommended_queue}</div>
                <div style="font-size:1.3rem;">≈ {rec_wait_min:.1f} min wait</div>
            </div>
            """,
            unsafe_allow_html=True,
        )
    else:
        st.info("No queue data yet.")

    st.write("")
    cols = st.columns(len(queue_names))
    for col, q in zip(cols, queue_names):
        with col:
            is_best = q == recommended_queue
            wait_min = wait_times.get(q, 0) / 60
            border = "3px solid #16a34a" if is_best else "1px solid #ddd"
            st.markdown(
                f"""
                <div style="padding:1rem;border-radius:0.75rem;border:{border};text-align:center;">
                    <div style="font-size:1rem;color:#555;">{q}</div>
                    <div style="font-size:1.8rem;font-weight:700;">{queue_counts[q]} 🧍</div>
                    <div style="font-size:0.95rem;color:#333;">≈ {wait_min:.1f} min</div>
                </div>
                """,
                unsafe_allow_html=True,
            )

# ---------------- Manager Dashboard ----------------
with tab_manager:
    st.subheader("Live status")

    rows = []
    for q in queue_names:
        counter_id = queue_to_counter[q]
        rows.append({
            "Queue": q,
            "Counter ID": counter_id,
            "People waiting": queue_counts[q],
            "Predicted service time (s/person)": round(service_times.get(counter_id, float("nan")), 1),
            "Estimated wait (min)": round(wait_times.get(q, 0) / 60, 2),
        })
    df = pd.DataFrame(rows)
    st.dataframe(df, use_container_width=True, hide_index=True)

    c1, c2 = st.columns(2)
    with c1:
        st.caption("People waiting per queue")
        st.bar_chart(df.set_index("Queue")["People waiting"])
    with c2:
        st.caption("Estimated wait per queue (min)")
        st.bar_chart(df.set_index("Queue")["Estimated wait (min)"])

    busiest = df.loc[df["People waiting"].idxmax()] if not df.empty else None
    if busiest is not None and busiest["People waiting"] >= alert_threshold:
        st.error(
            f"⚠️ MANAGER ALERT: {busiest['Queue']} has {int(busiest['People waiting'])} "
            f"people — consider opening another counter."
        )
    else:
        st.success("No congestion alert.")

    if recommended_queue:
        st.info(f"System would currently route customers to **{recommended_queue}**.")

st.divider()
with st.expander("Advanced: wire in live YOLO counts from a video (optional, needs ultralytics + opencv)"):
    st.markdown(
        """
        This tab only takes queue counts as numbers — it doesn't care where they come from.
        To feed it live counts from your CV pipeline instead of the manual sliders:

        1. Run your notebook's YOLO counting cell (the one that outputs a `queue_counts` dict)
           on a schedule (e.g. every 30 seconds) in a small background script.
        2. Have that script write the latest counts somewhere this app can read — the simplest
           options are a small JSON/CSV file, or a lightweight database/cache (e.g. Redis).
        3. In this file, replace the manual `st.slider` calls in the sidebar with code that
           reads that file/DB instead, e.g.:

        ```python
        import json
        with open("latest_counts.json") as f:
            queue_counts = json.load(f)
        ```

        Streamlit re-runs the whole script on each interaction, so pair this with
        `st.autorefresh` (from the `streamlit-autorefresh` package) or a manual
        "Refresh" button so the manager view updates without a full page reload.
        """
    )

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

your url is: https://good-cows-occur.loca.lt
2026-09-25 02:20:49.764 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.196.35.77:8501

2026-09-25 02:21:36.271 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-25 02:22:44.163 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-25 02:23:14.141 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width